# Amazon Customer Segmentation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

# 1. Load the data
Link for downloading data: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/YGLYDY

In [2]:
purchases = pd.read_csv("../data/raw/amazon-purchases.csv")
survey = pd.read_csv("../data/raw/survey.csv")

In [18]:
purchases.head(5)

,Order Date,Purchase Price Per Unit,Quantity,Shipping Address State,Title,ASIN/ISBN (Product Code),Category,Survey ResponseID
0,2018-12-04,7.98,1.0,NJ,SanDisk Ultra 16GB Class 10 SDHC UHS-I Memory ...,B0143RTB1E,FLASH_MEMORY,R_01vNIayewjIIKMF
1,2018-12-22,13.99,1.0,NJ,Betron BS10 Earphones Wired Headphones in Ear ...,B01MA1MJ6H,HEADPHONES,R_01vNIayewjIIKMF
2,2018-12-24,8.99,1.0,NJ,NaN,B078JZTFN3,NaN,R_01vNIayewjIIKMF
3,2018-12-25,10.45,1.0,NJ,Perfecto Stainless Steel Shaving Bowl. Durable...,B06XWF9HML,DISHWARE_BOWL,R_01vNIayewjIIKMF
4,2018-12-25,10.00,1.0,NJ,Proraso Shaving Cream for Men,B00837ZOI0,SHAVING_AGENT,R_01vNIayewjIIKMF


In [19]:
survey.head(5)

,Survey ResponseID,Q-demos-age,Q-demos-hispanic,Q-demos-race,Q-demos-education,Q-demos-income,Q-demos-gender,Q-sexual-orientation,Q-demos-state,Q-amazon-use-howmany,Q-amazon-use-hh-size,Q-amazon-use-how-oft,Q-substance-use-cigarettes,Q-substance-use-marijuana,Q-substance-use-alcohol,Q-personal-diabetes,Q-personal-wheelchair,Q-life-changes,Q-sell-YOUR-data,Q-sell-consumer-data,Q-small-biz-use,Q-census-use,Q-research-society
0,R_1ou69fj4DQGsVcp,35 - 44 years,No,Black or African American,High school diploma or GED,"$25,000 - $49,999",Female,heterosexual (straight),Iowa,2,2,Less than 5 times per month,Yes,No,Yes,No,No,Lost a job,No,No,No,No,No
1,R_2UbJL30HRjK1sdD,45 - 54 years,No,White or Caucasian,High school diploma or GED,"$100,000 - $149,999",Male,heterosexual (straight),Ohio,2,4+,Less than 5 times per month,No,No,No,No,No,NaN,No,No,No,No,Yes
2,R_UPXamGKtmf4RVIZ,25 - 34 years,No,White or Caucasian,High school diploma or GED,"$25,000 - $49,999",Male,heterosexual (straight),Arkansas,1 (just me!),2,Less than 5 times per month,No,No,No,Yes,No,NaN,No,No,No,No,Yes
3,R_2dYk5auG9Fv5Qve,35 - 44 years,Yes,White or Caucasian,"Graduate or professional degree (MA, MS, MBA, ...","$50,000 - $74,999",Male,heterosexual (straight),Tennessee,1 (just me!),1 (just me!),Less than 5 times per month,No,No,No,No,No,NaN,No,No,No,No,No
4,R_2aP0GyIR66gSTiR,25 - 34 years,No,White or Caucasian,High school diploma or GED,"$50,000 - $74,999",Male,heterosexual (straight),Virginia,2,3,Less than 5 times per month,No,No,Yes,No,No,NaN,No,Yes if consumers get part of the profit,I don't know,No,No


This project uses two primary datasets:

---

### a. Amazon Purchases Dataset (`amazon-purchases.csv`)

This dataset contains detailed information about customer purchase transactions on Amazon.  
It includes transactional and product-level attributes such as:

- Order Date  
- Purchase Price Per Unit  
- Quantity  
- Shipping Address State  
- Title (Product Name)  
- ASIN/ISBN (Product Code)  
- Category  
- Survey Response ID  

**Size**: 1,850,717 rows  

*Purpose*: This dataset is used to analyze customer purchasing behavior, product distribution, and geographic trends.

---

### b. Survey Dataset (`survey.csv`)

This dataset contains customer survey responses linked via `Survey ResponseID`.

It includes:
- Survey Response ID  
- 23 survey-related question columns  

**Size**: 5,027 rows × 23 columns  

*Purpose*: This dataset captures customer feedback and sentiment, which can be used to enrich behavioral analysis from purchase data.

---

Both datasets can be connected using the **Survey ResponseID**, enabling a combined view of:
- Customer behavior (purchases)
- Customer perception (survey responses)

In [17]:
print(purchases.shape)
print(survey.shape)

(1850717, 8)
(5027, 23)


### Check for missing values

In [23]:
missing_purchases = purchases.isnull().sum()
missing_survey = survey.isnull().sum()
print(missing_purchases)
print(missing_survey)

Order Date                      0
Purchase Price Per Unit         0
Quantity                        0
Shipping Address State      87812
Title                       89740
ASIN/ISBN (Product Code)      973
Category                    89458
Survey ResponseID               0
dtype: int64
Survey ResponseID                0
Q-demos-age                      0
Q-demos-hispanic                 0
Q-demos-race                     0
Q-demos-education                0
Q-demos-income                   0
Q-demos-gender                   0
Q-sexual-orientation             0
Q-demos-state                    0
Q-amazon-use-howmany             0
Q-amazon-use-hh-size             0
Q-amazon-use-how-oft             0
Q-substance-use-cigarettes       0
Q-substance-use-marijuana        0
Q-substance-use-alcohol          0
Q-personal-diabetes              0
Q-personal-wheelchair            0
Q-life-changes                3384
Q-sell-YOUR-data                 0
Q-sell-consumer-data             0
Q-small-biz-use

### a. Amazon Purchases Dataset (`amazon-purchases.csv`)

The dataset contains missing values in several key columns:

- **Shipping Address State**: 87,812 missing values  
- **Title**: 89,740 missing values  
- **ASIN/ISBN (Product Code)**: 973 missing values  
- **Category**: 89,458 missing values  

These missing values mainly appear in descriptive product and location fields, which may affect segmentation and analysis quality.

---

### b. Survey Dataset (`survey.csv`)

- No missing values detected (0 null values)


In [25]:
missing_pct_purchases = ((missing_purchases / len(purchases)) * 100).round(2)
print(pd.DataFrame({"missing_count": missing_purchases, "missing_pct": missing_pct_purchases}))

                          missing_count  missing_pct
Order Date                            0         0.00
Purchase Price Per Unit               0         0.00
Quantity                              0         0.00
Shipping Address State            87812         4.74
Title                             89740         4.85
ASIN/ISBN (Product Code)            973         0.05
Category                          89458         4.83
Survey ResponseID                     0         0.00


### Check for duplicates

In [3]:
print("Duplicate (purchases):", purchases.duplicated().sum())
print("Duplicate (survey):", survey["Survey ResponseID"].duplicated().sum())

Duplicate (purchases): 11624
Duplicate (survey): 0


In [4]:
# Remove duplicates
purchases = purchases.drop_duplicates()
print("Duplicate (purchases) after removal:", purchases.duplicated().sum())

Duplicate (purchases) after removal: 0


### Processing missing values

As Category and Shipping Address State are important for segmentation, missing columns are filled with "Unknown" to retain the records for analysis.


In [5]:
purchases["Category"] = purchases["Category"].fillna("Unknown")
purchases["Shipping Address State"] = purchases["Shipping Address State"].fillna("Unknown")

### Exploratory Data Analysis (EDA)

a. Date Range & Purchase Trends (annually)
- R in RFM (Recency, Frequency, Monetary) analysis, the date range of purchases is crucial for understanding customer behavior over time. The dataset spans from 2014 to 2018, allowing for trend analysis and seasonality detection.
- Today cannot be used as the reference date for recency calculations, as the dataset is historical. Instead, **the last purchase date in the dataset (2018-12-31)** will be used as the reference point for recency calculations.
b. Total Price distribution
- M in RFM analysis, the total price distribution helps identify high-value customers and outliers. The dataset shows a wide range of purchase amounts, with most transactions falling below $100, but some exceeding $1,000. This indicates a diverse customer base with varying spending habits.



In [6]:
# Date Range & Purchase Trends (annually)
purchases["Order Date"] = pd.to_datetime(purchases["Order Date"], errors="coerce")

print("Date Range of Purchases:", purchases["Order Date"].min(), "to", purchases["Order Date"].max())

snapshot_date = purchases["Order Date"].max()
print("Snapshot Date for Recency Calculation:", snapshot_date)

Date Range of Purchases: 2018-01-01 00:00:00 to 2024-08-15 00:00:00
Snapshot Date for Recency Calculation: 2024-08-15 00:00:00


In [ ]:
# Trend analysis: Count of purchases per year
purchases["year"] = purchases["Order Date"].dt.year
print(purchases["year"].value_counts().sort_index())

year
2018    224262
2019    264416
2020    406225
2021    464662
2022    439867
2023     39660
2024         1
Name: count, dtype: int64


In [ ]:
purchases["Total Price"] = purchases["Purchase Price Per Unit"] * purchases["Quantity"]

print(purchases["Total Price"].describe())
print("\nPercentile 95:", purchases["Total Price"].quantile(0.95))
print("Percentile 99:", purchases["Total Price"].quantile(0.99))

count    1.839093e+06
mean     2.385650e+01
std      4.929624e+01
min      1.000000e-02
25%      8.970000e+00
50%      1.481000e+01
75%      2.495000e+01
max      6.398950e+03
Name: Total Price, dtype: float64

Percentile 95: 64.99
Percentile 99: 182.99
